In [38]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from datetime import datetime

# LSTM Model Conceptualization

The following framework is followed for the LSTM model of the subway:

- Input Sequence: At each point in time (counted with 10 minute intervals), the following features are passed on:
    1. each station is placed as a categorical variable of whether or not a delay was recorded (this is as min delays is not a reliable measure for subway data)
    2. the hour and minute of the day (integer valued between 0-1440)
    3. the day of the week (integer valued between 0-6)
- Training: Data between 2016~2024-01-01 or 2020~2024-01-01 (the effectiveness of both should be tested and compared)
- Output Sequence: Given a certain sequence segment predict the next step in the sequence. We can utilize unseen data to do the testing for this

# Preprocess the Data

## Load the Data

In [39]:
file_name = "../data/cleaned_subway_data.csv"
out_data_name = "../data/lstm_data.csv"

sub_data = pd.read_csv(file_name)

## Convert data to sequential data of desired form

In [40]:
sub_data["10_min_time"] = pd.to_datetime(sub_data["Datetime"])
sub_data["10_min_time"] = sub_data["10_min_time"].dt.floor("10T")
fixed_sub_data = sub_data.groupby(["10_min_time", "Day","Station Name"])["Min Delay"].count().reset_index()
fixed_sub_data = pd.get_dummies(fixed_sub_data, columns=["Station Name"], drop_first=True)

day_conversion = {
    'Monday': 0,
    'Tuesday': 1,
    'Wednesday': 2,
    'Thursday': 3,
    'Friday': 4,
    'Saturday': 5,
    'Sunday': 6
}

fixed_sub_data['Day'] = fixed_sub_data['Day'].map(day_conversion)

/var/folders/xt/5my4_t657l5dvcsk9ybkb_ww0000gn/T/ipykernel_83267/3631191954.py:2: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  sub_data["10_min_time"] = sub_data["10_min_time"].dt.floor("10T")


In [41]:
date_range = pd.date_range(start="2016-01-01", end="2025-01-01", freq="10T")
dummy = pd.DataFrame({'10_min_time': date_range})

/var/folders/xt/5my4_t657l5dvcsk9ybkb_ww0000gn/T/ipykernel_83267/957377234.py:1: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  date_range = pd.date_range(start="2016-01-01", end="2025-01-01", freq="10T")


In [42]:
lstm_data = dummy.merge(fixed_sub_data, on='10_min_time', how='left')
station_cols = [col for col in lstm_data.columns if col.startswith('Station Name')]
lstm_data[station_cols] = lstm_data[station_cols].fillna(False)

lstm_data['10_min_time'] = pd.to_datetime(lstm_data['10_min_time'])
lstm_data['Day'] = lstm_data['10_min_time'].dt.dayofweek

lstm_data['Min Delay'] = lstm_data['Min Delay'].fillna(0)

display(lstm_data.head())

/var/folders/xt/5my4_t657l5dvcsk9ybkb_ww0000gn/T/ipykernel_83267/1756720721.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lstm_data[station_cols] = lstm_data[station_cols].fillna(False)


,10_min_time,Day,Min Delay,Station Name_BAY STATION,Station Name_BAYVIEW STATION,Station Name_BESSARION STATION,Station Name_BLOOR-YONGE STATION,Station Name_BROADVIEW STATION,Station Name_CASTLE FRANK STATION,Station Name_CHESTER STATION,...,Station Name_UNION STATION,Station Name_VAUGHAN METROPOLITAN CENTRE STATION,Station Name_VICTORIA PARK STATION,Station Name_WARDEN STATION,Station Name_WELLESLEY STATION,Station Name_WILSON STATION,Station Name_WOODBINE STATION,Station Name_YORK MILLS STATION,Station Name_YORK UNIVERSITY STATION,Station Name_YORKDALE STATION
0,2016-01-01 00:00:00,4,0.0,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,2016-01-01 00:10:00,4,0.0,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,2016-01-01 00:20:00,4,0.0,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,2016-01-01 00:30:00,4,0.0,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,2016-01-01 00:40:00,4,0.0,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [44]:
lstm_data.to_csv(out_data_name, index = False)

## Train/Test Splits

In [47]:
train_16_24 = lstm_data[lstm_data['10_min_time'] < '2024-01-01']
train_20_24 = train_16_24[train_16_24['10_min_time'] >= '2020-01-01']

test = lstm_data[lstm_data['10_min_time'] >= '2024-01-01']

# Define LSTM Model

## Set up instances of LSTM Model

# Train LSTM Model

# Test LSTM Model

# Conclusions and Discussion